# Latencia de extremo a extremo — Objetivo específico 4 / Hito H4

Mide el pipeline completo, de archivo de audio a archivo BRF, para verificar el criterio
del cuarto objetivo específico.

> El sistema debe entregar un BRF sintácticamente válido en un tiempo **menor o igual a
> 1.5 veces la duración del audio**, en al menos el **90 % de las ejecuciones**, para
> fragmentos de hasta cinco minutos.

Se cronometra cada etapa por separado, de modo que el resultado diga no solo si se cumple
sino dónde se consume el tiempo. La etapa determinista ya se midió en local y resulta
despreciable, del orden de 15 ms para cinco minutos de audio; esta corrida lo confirma
sobre audio real y añade la etapa acústica, que es la que domina.

El código se toma del repositorio para que la medición corresponda exactamente al sistema
entregado, sin reimplementaciones.

## 1. Repositorio y dependencias

In [1]:
import subprocess, sys, os
from pathlib import Path

REPO = "https://github.com/G2309/amt-braille-translator.git"
BRANCH = "dev"
WORK = Path("/kaggle/working/amt-braille-translator")

if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORK)], check=True)
sys.path.insert(0, str(WORK / "src"))

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("--no-deps", "piano_transcription_inference", "torchlibrosa", "mido")
pip("librosa", "soundfile")
print("commit:", subprocess.run(["git", "-C", str(WORK), "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

Cloning into '/kaggle/working/amt-braille-translator'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 1.4 MB/s eta 0:00:00
commit: e69fb54


## 2. Conjunto de prueba\n\nEl criterio aplica a fragmentos de hasta cinco minutos, así que se recortan segmentos de duración creciente sobre varias obras distintas. Así se comprueba además si el cociente se degrada con la duración.

In [2]:
import csv, glob, random
import librosa, soundfile as sf

DURACIONES = [30, 60, 120, 180, 300]   # segundos, dentro del alcance declarado
N_OBRAS = 4                            # obras distintas por duracion -> 20 ejecuciones
SEED = 22779

DATASET_ROOT = None
for cand in glob.glob("/kaggle/input/*/"):
    hits = glob.glob(cand + "**/maestro-v3.0.0.csv", recursive=True)
    if hits:
        DATASET_ROOT = Path(hits[0]).parent
        break
assert DATASET_ROOT is not None, "Montar el dataset the-maestro-dataset-v3-0-0 como input"

with open(DATASET_ROOT / "maestro-v3.0.0.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["split"] == "test"]

# solo obras suficientemente largas para recortar el segmento mayor
largas = [r for r in rows if float(r["duration"]) >= max(DURACIONES) + 5]
obras = random.Random(SEED).sample(largas, N_OBRAS)

def resolve(rel):
    p = DATASET_ROOT / rel
    if p.exists():
        return p
    hits = glob.glob(str(DATASET_ROOT / "**" / Path(rel).name), recursive=True)
    assert hits, f"No se encontro {rel}"
    return Path(hits[0])

SEGMENTOS = Path("/kaggle/working/segmentos")
SEGMENTOS.mkdir(exist_ok=True)

casos = []
for obra in obras:
    origen = resolve(obra["audio_filename"])
    for dur in DURACIONES:
        destino = SEGMENTOS / f"{Path(obra['midi_filename']).stem}__{dur}s.wav"
        if not destino.exists():
            audio, sr = librosa.load(str(origen), sr=None, mono=True, duration=dur)
            sf.write(str(destino), audio, sr)
        casos.append({"path": destino, "duracion_objetivo": dur,
                      "compositor": obra["canonical_composer"],
                      "obra": obra["canonical_title"]})

print(f"{len(casos)} segmentos preparados sobre {N_OBRAS} obras")

20 segmentos preparados sobre 4 obras


## 3. Medición\n\nEl modelo se carga una sola vez antes de cronometrar, para que el tiempo de carga del checkpoint no se cuente como latencia de proceso.

In [3]:
import torch
from amt.transcriber import AMTTranscriber
from evaluation.latency import LatencySummary, measure_end_to_end

device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)

transcriptor = AMTTranscriber(device=device)
# calentamiento: descarga y carga del checkpoint fuera de la medicion
transcriptor.transcribe(str(casos[0]["path"]))
print("modelo listo")

dispositivo: cuda
Checkpoint path: /root/piano_transcription_inference_data/note_F1=0.9677_pedal_F1=0.9186.pth
Total size: ~165 MB


--2026-09-26 03:10:36--  https://zenodo.org/record/4034264/files/CRNN_note_F1%3D0.9677_pedal_F1%3D0.9186.pth?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 188.185.43.153, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/4034264/files/CRNN_note_F1=0.9677_pedal_F1=0.9186.pth [following]
--2026-09-26 03:10:37--  https://zenodo.org/records/4034264/files/CRNN_note_F1=0.9677_pedal_F1=0.9186.pth
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 171966578 (164M) [application/octet-stream]
Saving to: ‘/root/piano_transcription_inference_data/note_F1=0.9677_pedal_F1=0.9186.pth’

     0K .......... .......... .......... .......... ..........  0%  204K 13m41s
    50K .......... .......... .......... .......... ..........  0%  411K 10m15s
   100K .......... .......... .......... .......... ..........  0%  1

Using cuda for inference.
GPU number: 2
Segment 0 / 5
Segment 1 / 5
Segment 2 / 5
Segment 3 / 5
Segment 4 / 5
Segment 5 / 5
modelo listo


In [4]:
SALIDAS = Path("/kaggle/working/brf")
SALIDAS.mkdir(exist_ok=True)

runs, filas = [], []
for caso in casos:
    destino_brf = SALIDAS / (caso["path"].stem + ".brf")
    run = measure_end_to_end(str(caso["path"]), transcriptor.transcribe,
                             output_path=str(destino_brf))
    runs.append(run)
    fila = run.as_dict()
    fila.update({"duracion_objetivo": caso["duracion_objetivo"],
                 "compositor": caso["compositor"], "obra": caso["obra"],
                 "brf": str(destino_brf)})
    filas.append(fila)
    print(f"{caso['duracion_objetivo']:>4}s  {caso['compositor'][:18]:18s} "
          f"amt={run.timings.amt_s:6.2f}s  det={run.timings.deterministic_s:6.3f}s  "
          f"total={run.timings.total_s:6.2f}s  x={run.ratio:.4f}  "
          f"{'ok' if run.within_threshold else 'EXCEDE'}")

Segment 0 / 5
Segment 1 / 5
Segment 2 / 5
Segment 3 / 5
Segment 4 / 5
Segment 5 / 5
  30s  Joseph Haydn       amt=  2.12s  det= 0.002s  total=  2.12s  x=0.0708  ok
Segment 0 / 11
Segment 1 / 11
Segment 2 / 11
Segment 3 / 11
Segment 4 / 11
Segment 5 / 11
Segment 6 / 11
Segment 7 / 11
Segment 8 / 11
Segment 9 / 11
Segment 10 / 11
Segment 11 / 11
  60s  Joseph Haydn       amt=  4.62s  det= 0.003s  total=  4.62s  x=0.0770  ok
Segment 0 / 23
Segment 1 / 23
Segment 2 / 23
Segment 3 / 23
Segment 4 / 23
Segment 5 / 23
Segment 6 / 23
Segment 7 / 23
Segment 8 / 23
Segment 9 / 23
Segment 10 / 23
Segment 11 / 23
Segment 12 / 23
Segment 13 / 23
Segment 14 / 23
Segment 15 / 23
Segment 16 / 23
Segment 17 / 23
Segment 18 / 23
Segment 19 / 23
Segment 20 / 23
Segment 21 / 23
Segment 22 / 23
Segment 23 / 23
 120s  Joseph Haydn       amt=  9.72s  det= 0.007s  total=  9.73s  x=0.0811  ok
Segment 0 / 35
Segment 1 / 35
Segment 2 / 35
Segment 3 / 35
Segment 4 / 35
Segment 5 / 35
Segment 6 / 35
Segment 7 / 35


## 4. Verificación del criterio

In [5]:
resumen = LatencySummary(runs)
print(resumen.report())

Ejecuciones            : 20
Dentro de 1.5x         : 100.0% (exigido 90%) -> CUMPLE
Cociente mediano       : 0.0813
Cociente p90           : 0.0832
Cociente maximo        : 0.0844

Reparto del tiempo total por etapa
  amt         99.90%
  quantize     0.05%
  translate    0.04%
  render       0.00%
  export       0.01%


## 5. Validez sintáctica de los BRF generados\n\nEl objetivo exige que el archivo entregado sea sintácticamente válido, no solo que llegue a tiempo.

In [6]:
from braille_translator.brf_exporter import validate_brf

invalidos = 0
for fila in filas:
    problemas = validate_brf(fila["brf"])
    fila["brf_valido"] = not problemas
    fila["brf_problemas"] = "; ".join(problemas)
    if problemas:
        invalidos += 1
        print(f"{Path(fila['brf']).name}: {problemas[:3]}")

print(f"\nBRF validos: {len(filas) - invalidos}/{len(filas)}")


BRF validos: 20/20


## 6. Resultados por duración y exportación

In [7]:
import pandas as pd

df = pd.DataFrame(filas)
por_duracion = df.groupby("duracion_objetivo").agg(
    ejecuciones=("ratio", "count"),
    amt_s=("amt_s", "mean"),
    deterministic_s=("deterministic_s", "mean"),
    total_s=("total_s", "mean"),
    ratio_medio=("ratio", "mean"),
    ratio_max=("ratio", "max"),
).round(4)
display(por_duracion)

df.to_csv("/kaggle/working/latencia_e2e.csv", index=False)
por_duracion.to_csv("/kaggle/working/latencia_por_duracion.csv")
print("CSV escritos")

,ejecuciones,amt_s,deterministic_s,total_s,ratio_medio,ratio_max
duracion_objetivo,,,,,,
30,4,2.1392,0.0024,2.1417,0.0714,0.0727
60,4,4.7324,0.0045,4.7369,0.0789,0.0809
120,4,9.7731,0.0101,9.7832,0.0815,0.0827
180,4,14.8164,0.0144,14.8307,0.0824,0.0831
300,4,25.0502,0.0249,25.0751,0.0836,0.0844


CSV escritos


## 7. Trazabilidad

- El criterio verificado corresponde al cuarto objetivo específico y al hito H4.
- La medición usa el código del repositorio en el commit impreso en la celda 1, de modo
  que es reproducible y corresponde al sistema entregado.
- `latencia_e2e.csv` guarda una fila por ejecución con el desglose por etapa; con eso se
  redacta la sección de resultados sin volver a ejecutar nada.